# <font color="darkblue"> Prática 02: Regressão Logística - Ataque de Coração</font>

**Objetivos:**


*   Apresentar  plataforma Kaglee
*   Inferir dados de uma base real utilizando o algoritmo de Regressão Logística 

**Requisitos de execução:**


*   Upload dos arquivos *logisticregression.py* e *heart_failure_clinical_records_dataset.csv*

**Atividade 1:**

1. Visitar a base de dados: https://www.kaggle.com/andrewmvd/heart-failure-clinical-data
2. Carregar os dados do arquivo *heart_failure_clinical_records_dataset.csv* utilizando o pandas.

    

In [ ]:
import pandas as pd
import numpy as np

heart_data = pd.read_csv('heart_failure_clinical_records_dataset.csv')
print(heart_data.head())


    age  anaemia  creatinine_phosphokinase  diabetes  ejection_fraction  \
0  75.0        0                       582         0                 20   
1  55.0        0                      7861         0                 38   
2  65.0        0                       146         0                 20   
3  50.0        1                       111         0                 20   
4  65.0        1                       160         1                 20   

   high_blood_pressure  platelets  serum_creatinine  serum_sodium  sex  \
0                    1  265000.00               1.9           130    1   
1                    0  263358.03               1.1           136    1   
2                    0  162000.00               1.3           129    1   
3                    0  210000.00               1.9           137    1   
4                    0  327000.00               2.7           116    0   

   smoking  time  DEATH_EVENT  
0        0     4            1  
1        0     6            1  
2       

**Atividade 2:**

1. Extrair os valores do *DataFrame* pandas e colocar nas variáveis


In [6]:
from sklearn.model_selection import train_test_split

Features = [
    'time', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
    'ejection_fraction', 'serum_creatinine', 'age',
    'high_blood_pressure', 'platelets', 'serum_sodium',
    'sex', 'smoking'
]

x = heart_data[Features].values
y = heart_data["DEATH_EVENT"].values

# split ANTES de qualquer transformação
x_T, x_test, y_T, y_test = train_test_split(
    x, y,
    test_size=0.25,
    random_state=0
)

**Atividade 3:**

1. Separar os dados em conjunto de treinamento e teste


In [7]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()

# aprende APENAS no treino
sc.fit(x_T)

# aplica transformação
x_T = sc.transform(x_T)
x_test = sc.transform(x_test)

print("Tamanho treinamento:", len(x_T))
print("Tamanho teste:", len(x_test))

Tamanho treinamento: 224
Tamanho teste: 75


**Atividade 4:**

1. Inferir a função a função hipótese $g(x)=\theta(w^Tx)$ dos dados de treinamento;
2. Computar o erro dentro da amostra ($E_{in}$);
3. Predizer os dados de teste usando $g(x)$;
4. Computar o erro fora da amostra ($E_{out}$);
5. Computar as métricas de aprendizado sobre o dados de teste.
6. Aplique a normalização dos dados de entrada e reexecute todos os experimentos. Compare os resultados.

In [8]:
from logisticregression import LogisticRegression
from sklearn.metrics import classification_report

# converter y para {-1, +1}
lrY = [+1 if value == 1 else -1 for value in y_T]
lrY_test = [+1 if value == 1 else -1 for value in y_test]

# modelo
classifier = LogisticRegression(0.1, 3000, 32)
classifier.fit(x_T, lrY)

# -------------------------
# Ein (erro in-sample)
# -------------------------
train_pred = classifier.predict(x_T)

Ein = sum(train_pred[i] != lrY[i] for i in range(len(lrY))) / len(lrY)
print("Ein =", Ein)

# -------------------------
# Eout (erro out-of-sample)
# -------------------------
test_pred = classifier.predict(x_test)

Eout = sum(test_pred[i] != lrY_test[i] for i in range(len(lrY_test))) / len(lrY_test)
print("Eout =", Eout)

# relatório sklearn (convertendo para 0/1 para evitar confusão)
print(classification_report(lrY_test, test_pred))

Ein = 0.13392857142857142
Eout = 0.25333333333333335
              precision    recall  f1-score   support

          -1       0.75      0.92      0.82        48
           1       0.75      0.44      0.56        27

    accuracy                           0.75        75
   macro avg       0.75      0.68      0.69        75
weighted avg       0.75      0.75      0.73        75



**Atividade 5:**

1. Reproduza o mesmo experimento com a classe LogisticRegression do pacote *sklearn.metrics*

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# -------------------------
# 1. Features e dados
# -------------------------
Features = [
    'time', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
    'ejection_fraction', 'serum_creatinine', 'age',
    'high_blood_pressure', 'platelets', 'serum_sodium',
    'sex', 'smoking'
]

x = heart_data[Features].values
y = heart_data["DEATH_EVENT"].values  # sklearn usa 0/1

# -------------------------
# 2. Split (primeiro passo SEMPRE)
# -------------------------
x_T, x_test, y_T, y_test = train_test_split(
    x, y,
    test_size=0.25,
    random_state=0
)

# -------------------------
# 3. Normalização correta (sem leakage)
# -------------------------
sc = StandardScaler()

sc.fit(x_T)

x_T = sc.transform(x_T)
x_test = sc.transform(x_test)

# -------------------------
# 4. Logistic Regression (sklearn)
# -------------------------
model = LogisticRegression(max_iter=1000)

model.fit(x_T, y_T)

# -------------------------
# 5. Predição e avaliação
# -------------------------
y_pred = model.predict(x_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.77      0.92      0.84        48
           1       0.78      0.52      0.62        27

    accuracy                           0.77        75
   macro avg       0.77      0.72      0.73        75
weighted avg       0.77      0.77      0.76        75



**Atividade 6:**

1. Exiba os parâmetros (Features) de $X$ ordenados pela sua importância na função de decisão, este valor é indicado pelo vetor $w$. No pacote sklearn.linear_model.LogisticRegression, utilize o atributo $coef_$.

In [10]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# coeficientes do modelo sklearn
coef = model.coef_[0]

# emparelhar features com coeficientes
param = [(coef[i], Features[i]) for i in range(len(Features))]

# ordenar pela importância (maior coeficiente primeiro)
param.sort(reverse=True)

# imprimir resultado
for p in param:
    print(p)

(np.float64(0.705453511516202), 'age')
(np.float64(0.3968557881748854), 'serum_creatinine')
(np.float64(0.3920201594528659), 'creatinine_phosphokinase')
(np.float64(0.1415001610559574), 'diabetes')
(np.float64(0.09161395400559845), 'smoking')
(np.float64(0.06574804131946958), 'anaemia')
(np.float64(-0.118729055084082), 'platelets')
(np.float64(-0.15074370045625454), 'high_blood_pressure')
(np.float64(-0.3127106941751498), 'serum_sodium')
(np.float64(-0.44777851677955965), 'sex')
(np.float64(-0.9533105447030003), 'ejection_fraction')
(np.float64(-1.5146810687718737), 'time')
